# Benchmark and Compare RTMPose Predictions

This notebook provides a lightweight wrapper around `scripts/post_process/benchmark.py` and `scripts/post_process/compare_models.py`.

Use it to evaluate a single model prediction JSON and to compare two prediction JSONs side-by-side.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Detect repository root by walking up until 'scripts/post_process/benchmark.py' exists.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent:
    if (repo_root / 'scripts' / 'post_process' / 'benchmark.py').exists():
        break
    repo_root = repo_root.parent
else:
    raise FileNotFoundError('Could not locate scripts/post_process/benchmark.py from the current working directory.')

benchmark_script = repo_root / 'scripts' / 'post_process' / 'benchmark.py'
compare_script = repo_root / 'scripts' / 'post_process' / 'compare_models.py'
labels_json = repo_root / 'input' / 'labels' / 'merged.json'

print(f'Repo root: {repo_root}')
print(f'Benchmark script: {benchmark_script}')
print(f'Compare script: {compare_script}')
print(f'Labels file: {labels_json}')

## Configure prediction file paths

Set the prediction JSON files to evaluate and compare. Update the paths below to match your output locations.

In [ ]:
# Update these paths for your run outputs.
benchmark_prediction = repo_root / 'output' / 'RTMPose' / 'finetune' / 'predictions.json'
compare_old_prediction = repo_root / 'output' / 'RTMPose' / 'baseline' / 'predictions.json'
compare_new_prediction = repo_root / 'output' / 'RTMPose' / 'finetune' / 'predictions.json'

print('Benchmark prediction path:', benchmark_prediction)
print('Compare old prediction path:', compare_old_prediction)
print('Compare new prediction path:', compare_new_prediction)

## Run benchmark on one model

This executes `benchmark.py` and prints the summary plus plot generation messages.

In [ ]:
def run_command(cmd):
    print('Running:', ' '.join(str(x) for x in cmd))
    result = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    result.check_returncode()

run_command([
    sys.executable,
    str(benchmark_script),
    str(benchmark_prediction),
    '--labels',
    str(labels_json),
    '--name',
    'Finetune model',
])

## Compare two models

This executes `compare_models.py` for two prediction JSON files and prints the comparison summary.

In [ ]:
run_command([
    sys.executable,
    str(compare_script),
    str(compare_old_prediction),
    str(compare_new_prediction),
    '--labels',
    str(labels_json),
    '--names',
    'Baseline',
    'Finetune',
])